In [6]:
# imports and env
import base64
from dotenv import load_dotenv
from langchain_unstructured.document_loaders import UnstructuredLoader
from langchain_openai import ChatOpenAI
from langchain_experimental.open_clip import OpenCLIPEmbeddings
from langchain_ollama.embeddings import OllamaEmbeddings
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import PydanticOutputParser
from langchain_chroma import Chroma
from langchain_community.vectorstores.utils import filter_complex_metadata
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.messages import HumanMessage

load_dotenv()

False

In [7]:
import open_clip

open_clip.list_pretrained()

/Users/itaakash/Desktop/Lakhan/Tasks/ragAdancedCampusX/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[('RN50', 'openai'),
 ('RN50', 'yfcc15m'),
 ('RN50', 'cc12m'),
 ('RN101', 'openai'),
 ('RN101', 'yfcc15m'),
 ('RN50x4', 'openai'),
 ('RN50x16', 'openai'),
 ('RN50x64', 'openai'),
 ('ViT-B-32', 'openai'),
 ('ViT-B-32', 'laion400m_e31'),
 ('ViT-B-32', 'laion400m_e32'),
 ('ViT-B-32', 'laion2b_e16'),
 ('ViT-B-32', 'laion2b_s34b_b79k'),
 ('ViT-B-32', 'datacomp_xl_s13b_b90k'),
 ('ViT-B-32', 'datacomp_m_s128m_b4k'),
 ('ViT-B-32', 'commonpool_m_clip_s128m_b4k'),
 ('ViT-B-32', 'commonpool_m_laion_s128m_b4k'),
 ('ViT-B-32', 'commonpool_m_image_s128m_b4k'),
 ('ViT-B-32', 'commonpool_m_text_s128m_b4k'),
 ('ViT-B-32', 'commonpool_m_basic_s128m_b4k'),
 ('ViT-B-32', 'commonpool_m_s128m_b4k'),
 ('ViT-B-32', 'datacomp_s_s13m_b4k'),
 ('ViT-B-32', 'commonpool_s_clip_s13m_b4k'),
 ('ViT-B-32', 'commonpool_s_laion_s13m_b4k'),
 ('ViT-B-32', 'commonpool_s_image_s13m_b4k'),
 ('ViT-B-32', 'commonpool_s_text_s13m_b4k'),
 ('ViT-B-32', 'commonpool_s_basic_s13m_b4k'),
 ('ViT-B-32', 'commonpool_s_s13m_b4k'),
 ('ViT-

In [8]:
# load PDF
PDF_PATH = "../data/crag_paper.pdf"

loader = UnstructuredLoader(
    PDF_PATH,
    mode="elements",
    strategy="hi_res",
    extract_images_in_pdf=True,
)
elements = loader.load()

print(f"Loaded {len(elements)} elements")
for cat in sorted(set(el.metadata.get("category", "unknown") for el in elements)):
    count = sum(1 for el in elements if el.metadata.get("category") == cat)
    print(f"  {cat}: {count}")

INFO: pikepdf C++ to Python logger bridge initialized
INFO: HTTP Request: HEAD https://huggingface.co/unstructuredio/yolo_x_layout/resolve/main/yolox_l0.05.onnx "HTTP/1.1 302 Found"
INFO: Reading PDF for file: ../data/crag_paper.pdf ...


Loaded 268 elements
  FigureCaption: 6
  Footer: 1
  Formula: 1
  Header: 1
  Image: 8
  ListItem: 41
  NarrativeText: 107
  Table: 7
  Title: 32
  UncategorizedText: 64


In [9]:
# split text
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

text_elements = [el for el in elements if el.metadata.get("category") != "Image"]
text_docs = splitter.split_documents(text_elements)
print(f"Text chunks: {len(text_docs)}")

Text chunks: 269


In [10]:
# CLIP embeddings
clip_embeddings = OpenCLIPEmbeddings(model_name="ViT-B-32", checkpoint="laion2b_s34b_b79k")

INFO: Parsing model identifier. Schema: None, Identifier: ViT-B-32
INFO: Loaded built-in ViT-B-32 model config.
INFO: HTTP Request: HEAD https://huggingface.co/laion/CLIP-ViT-B-32-laion2B-s34B-b79K/resolve/main/open_clip_model.safetensors "HTTP/1.1 302 Found"
INFO: HTTP Request: GET https://huggingface.co/api/models/laion/CLIP-ViT-B-32-laion2B-s34B-b79K/xet-read-token/1a25a446712ba5ee05982a381eed697ef9b435cf "HTTP/1.1 200 OK"
INFO: Instantiating model architecture: CLIP
INFO: Loading full pretrained weights from: /Users/itaakash/.cache/huggingface/hub/models--laion--CLIP-ViT-B-32-laion2B-s34B-b79K/snapshots/1a25a446712ba5ee05982a381eed697ef9b435cf/open_clip_model.safetensors
INFO: Final image preprocessing configuration set: {'size': (224, 224), 'mode': 'RGB', 'mean': (0.48145466, 0.4578275, 0.40821073), 'std': (0.26862954, 0.26130258, 0.27577711), 'interpolation': 'bicubic', 'resize_mode': 'shortest', 'fill_color': 0}
INFO: Model ViT-B-32 creation process complete.
INFO: Parsing token

In [11]:
# vector store
store = Chroma(embedding_function=clip_embeddings, collection_name="multimodal")
text_ids = store.add_documents(filter_complex_metadata(text_docs))
print(f"Added {len(text_ids)} text chunks")

Added 269 text chunks


In [12]:
# add images
image_elements = [
    el for el in elements
    if el.metadata.get("category") == "Image" and el.metadata.get("image_path")
]

image_uris = [el.metadata["image_path"] for el in image_elements]
image_metadatas = [doc.metadata for doc in filter_complex_metadata(image_elements)]

image_ids = store.add_images(uris=image_uris, metadatas=image_metadatas)
print(f"Added {len(image_ids)} images")

Added 8 images


In [13]:
image_ids

['a2f8d6ed-141d-41c9-b0b1-5ab4611b8f3b',
 'deab7ab3-405b-4558-bb2e-cb361d9f239e',
 '6a1f3ca6-1cb8-4893-8a64-d764d2022864',
 '41024b22-31fb-4c9c-b6d7-c32bd4f6876b',
 'd0bbfaa6-3b41-4873-95d6-dfac8268ddb4',
 '7a96e58a-0fbf-4957-ada0-7faf9058824f',
 'cc9ef668-c9aa-455a-96bf-0e5d1e942432',
 '18fbe74f-1d47-4e2a-aec0-3eeb604bcd12']

In [14]:
# retriever
retriever = store.as_retriever(search_kwargs={"k": 6})

In [15]:
# RAG chain
def encode_image(image_path: str) -> str:
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def build_messages(inputs):
    docs = inputs["docs"]
    question = inputs["question"]

    text_chunks = [d for d in docs if d.metadata.get("category") != "Image"]
    image_chunks = [d for d in docs if d.metadata.get("category") == "Image"]

    text_context = "\n\n".join(d.page_content for d in text_chunks)

    content = [
        {"type": "text", "text": f"Answer the question based only on the following context:\n\n{text_context}\n\nQuestion: {question}"},
    ]

    seen = set()
    for doc in image_chunks:
        path = doc.page_content  # add_images stores URI as page_content
        if path and path not in seen:
            seen.add(path)
            content.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{encode_image(path)}"}})

    return [HumanMessage(content=content)]

llm = ChatOllama(model="qwen3.5:9b")

chain = (
    {"docs": retriever, "question": RunnablePassthrough()}
    | RunnableLambda(build_messages)
    | llm
)

In [16]:
# test
question = "How does generation accuracy change as retrieval accuracy drops for Self-CRAG and Self-RAG. explain in detail?"
answer = chain.invoke(question)
print(answer)

INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


content='Based on the provided context, the relationship between generation accuracy and retrieval accuracy changes as follows:\n\n*   **General Trend:** The generation performance of both Self-RAG and Self-CRAG dropped as the retrieval performance dropped. This indicates that the generator relies heavily on the quality of the retriever.\n*   **Comparison:** As retrieval performance dropped, the generation performance of Self-CRAG dropped more slightly than that of Self-RAG.\n*   **Implication:** These results imply the superiority of Self-CRAG over Self-RAG in enhancing robustness to retrieval performance.' additional_kwargs={} response_metadata={'model': 'qwen3.5:9b', 'created_at': '2026-11-23T10:25:25.503211Z', 'done': True, 'done_reason': 'stop', 'total_duration': 84929196709, 'load_duration': 19443917, 'prompt_eval_count': 385, 'prompt_eval_duration': 4320053458, 'eval_count': 1311, 'eval_duration': 80577682041, 'logprobs': None, 'model_name': 'qwen3.5:9b', 'model_provider': 'olla

In [17]:
question = "Computational requirements of CRAG vs Self-RAG and which was has faster execution time and can you give me the actual TFLOPS values?"
answer = chain.invoke(question)
print(answer)

INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


content='Based on the provided context table:\n\n*   **CRAG:** TFLOPs per token is **27.2** and executing time is **0.512s**.\n*   **Self-RAG:** TFLOPs per token ranges from **26.5∼132.4** and executing time is **0.741s**.\n\n**Execution Time Comparison:** CRAG has the faster execution time (**0.512s**) compared to Self-RAG (**0.741s**).' additional_kwargs={} response_metadata={'model': 'qwen3.5:9b', 'created_at': '2026-11-23T10:34:38.625699Z', 'done': True, 'done_reason': 'stop', 'total_duration': 150472387292, 'load_duration': 21273709, 'prompt_eval_count': 526, 'prompt_eval_duration': 5580677834, 'eval_count': 2229, 'eval_duration': 144857966333, 'logprobs': None, 'model_name': 'qwen3.5:9b', 'model_provider': 'ollama'} id='lc_run--01a207f6-8cb8-775d-92fa-631953658e65-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 526, 'output_tokens': 2229, 'total_tokens': 2755}
